# htmlsight — Treinamento no Google Colab

Detector visual multi-task de componentes web Bootstrap.

**Antes de começar:**
1. Vá em `Ambiente de execução → Alterar tipo de ambiente de execução`
2. Selecione **GPU → T4** (gratuita) ou **A100** (Colab Pro)
3. Execute as células em ordem

## 1. Verificar GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {gpu} ({mem:.1f} GB)')
else:
    print('⚠️  Sem GPU. Treine com epochs=1 para teste, ou ative a GPU acima.')

## 2. Montar Google Drive (opcional — persiste dataset e pesos)

Pule esta célula se não quiser salvar no Drive. Sem o Drive, tudo é perdido ao fechar o Colab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/htmlsight'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive montado. Dados serão salvos em: {DRIVE_DIR}')

## 3. Clonar repositório e instalar dependências

In [ ]:
%cd /content
!git clone https://github.com/LucasMe110/htmlsight.git
%cd htmlsight

In [ ]:
# Instalar dependências diretamente no ambiente do Colab (sem venv)
!pip install -q -e '.[dev,render,model]'

# Instalar Playwright e baixar Chromium
!pip install -q playwright
!playwright install chromium
!playwright install-deps chromium

print('✅ Dependências instaladas')

In [ ]:
# Verificar instalação
import sys
sys.path.insert(0, 'src')

import torch, ultralytics, playwright
print(f'torch:        {torch.__version__}')
print(f'ultralytics:  {ultralytics.__version__}')
print(f'CUDA:         {torch.cuda.is_available()}')

# Rodar testes para confirmar que tudo está ok
!python -m pytest -q --tb=short 2>&1 | tail -5

## 4. Configurar caminhos

Edite `USE_DRIVE = True` se montou o Google Drive na célula 2.

In [ ]:
import os
from pathlib import Path

USE_DRIVE = False  # mude para True se montou o Google Drive

if USE_DRIVE:
    BASE = Path('/content/drive/MyDrive/htmlsight')
else:
    BASE = Path('/content/htmlsight')

DATASET_DIR = BASE / 'data/dataset'
RUNS_DIR    = BASE / 'runs'
REPORT_DIR  = BASE / 'runs/baseline/report'

os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

print(f'Dataset:  {DATASET_DIR}')
print(f'Runs:     {RUNS_DIR}')

## 5. Gerar dataset

Escolha o modo:
- **`SYNTHETIC_ONLY = True`** → rápido (~30s), sem Chromium, bom para testar o pipeline
- **`SYNTHETIC_ONLY = False`** → dataset real com Playwright (~15–20 min para 3000 imgs)

In [ ]:
SYNTHETIC_ONLY = False  # True = rápido/sintético | False = real com Playwright
NUM_IMAGES     = 3000   # recomendado para treino real; use 50-100 para testar
NUM_WORKERS    = 4      # workers paralelos para Playwright

import subprocess, sys

cmd = [
    sys.executable, '-m', 'ia_visao_web.cli', 'dataset', 'build',
    '--count', str(NUM_IMAGES),
    '--workers', str(NUM_WORKERS),
    '--output', str(DATASET_DIR),
]
if SYNTHETIC_ONLY:
    cmd.append('--synthetic-only')

env = os.environ.copy()
env['PYTHONPATH'] = 'src'

print(f'Gerando {NUM_IMAGES} imagens ({"sintético" if SYNTHETIC_ONLY else "Playwright"})...')
result = subprocess.run(cmd, env=env, cwd='/content/htmlsight')

imgs = list(Path(DATASET_DIR).glob('images/**/*.png'))
print(f'\n✅ {len(imgs)} imagens geradas em {DATASET_DIR}')

## 6. Validar dataset

In [ ]:
import subprocess, sys, os

result = subprocess.run(
    [sys.executable, '-m', 'ia_visao_web.cli', 'dataset', 'validate',
     '--root', str(DATASET_DIR), '--report', '--min-train-instances', '10'],
    env={**os.environ, 'PYTHONPATH': 'src'},
    cwd='/content/htmlsight',
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

# Mostrar distribuição de classes
import json
report_path = DATASET_DIR / '_qa/report.json'
if report_path.exists():
    r = json.loads(report_path.read_text())
    counts = r.get('class_counts', {}).get('train', {})
    print('\nInstâncias por classe (train):')
    for cls, n in sorted(counts.items(), key=lambda x: -x[1]):
        bar = '█' * min(30, n // 500)
        print(f'  {cls:12s} {n:6d}  {bar}')

## 7. Treinar modelo

Com GPU T4 (~16GB), 100 épocas levam ~2–3 horas para 3000 imagens.

In [ ]:
EPOCHS     = 100  # use 1 para smoke test rápido
BATCH_SIZE = 16   # reduza para 8 se der OOM
IMAGE_SIZE = 640

import subprocess, sys, os

result = subprocess.run(
    [
        sys.executable, '-m', 'ia_visao_web.cli', 'train',
        '--dataset', str(DATASET_DIR),
        '--output',  str(RUNS_DIR / 'baseline'),
        '--epochs',  str(EPOCHS),
        '--batch-size', str(BATCH_SIZE),
        '--image-size', str(IMAGE_SIZE),
        '--device', 'cuda' if torch.cuda.is_available() else 'cpu',
    ],
    env={**os.environ, 'PYTHONPATH': 'src'},
    cwd='/content/htmlsight',
)

weights = RUNS_DIR / 'baseline/weights/best.pt'
if weights.exists():
    print(f'\n✅ Treino concluído! Pesos salvos em: {weights}')
else:
    print('⚠️  Pesos não encontrados. Verifique os logs acima.')

## 8. Gerar relatório de métricas

In [ ]:
import subprocess, sys, os

weights = RUNS_DIR / 'baseline/weights/best.pt'

if not weights.exists():
    print('⚠️  Treine o modelo primeiro (célula 7)')
else:
    subprocess.run(
        [
            sys.executable, '-m', 'ia_visao_web.cli', 'report',
            '--dataset', str(DATASET_DIR),
            '--weights', str(weights),
            '--output',  str(REPORT_DIR),
            '--split', 'test',
        ],
        env={**os.environ, 'PYTHONPATH': 'src'},
        cwd='/content/htmlsight',
    )

    # Exibir relatório Markdown
    from IPython.display import Markdown, display
    md_path = REPORT_DIR / 'report.md'
    if md_path.exists():
        display(Markdown(md_path.read_text()))

## 9. Predição em imagem

In [ ]:
import subprocess, sys, os, json
from pathlib import Path
from IPython.display import Image as IPyImage, display

# Escolha uma imagem do test set para testar
test_images = sorted(Path(DATASET_DIR / 'images/test').glob('*.png'))
if not test_images:
    print('Sem imagens no test set.')
else:
    img_path = test_images[0]
    weights  = RUNS_DIR / 'baseline/weights/best.pt'

    cmd = [
        sys.executable, '-m', 'ia_visao_web.cli', 'predict', str(img_path),
    ]
    if weights.exists():
        cmd += ['--weights', str(weights)]
    else:
        print('⚠️  Sem pesos treinados — retornando detections: [] (stub)')

    result = subprocess.run(
        cmd,
        env={**os.environ, 'PYTHONPATH': 'src'},
        cwd='/content/htmlsight',
        capture_output=True, text=True
    )

    display(IPyImage(filename=img_path, width=640))
    print('\nDetecções:')
    print(result.stdout)

## 10. Baixar pesos e relatório

Se não usou o Google Drive, baixe os arquivos antes de fechar o Colab.

In [ ]:
from google.colab import files
from pathlib import Path

to_download = [
    RUNS_DIR / 'baseline/weights/best.pt',
    REPORT_DIR / 'report.md',
    REPORT_DIR / 'report.json',
]

for f in to_download:
    if Path(f).exists():
        print(f'Baixando {f.name}...')
        files.download(str(f))
    else:
        print(f'⚠️  Não encontrado: {f}')